<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/01_pandas_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: pandas Fundamentals

> **Goal:** Master the core pandas operations you'll use in every data science project.

**Dataset:** Titanic (891 passengers)

**What you'll learn:**
1. Loading and inspecting data
2. Selecting, filtering, and sorting
3. GroupBy aggregations
4. Handling missing values
5. Feature engineering
6. Merging DataFrames

**Time:** ~2 hours

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Nicer display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

## Part 1: Loading and Inspecting Data

In [ ]:
# Load Titanic dataset
URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(URL)

print(f'Shape: {df.shape}')  # (rows, columns)
print(f'\nColumn names: {df.columns.tolist()}')

In [ ]:
# First look
df.head()

In [ ]:
# Data types and memory
df.info()

In [ ]:
# Statistical summary for numeric columns
df.describe()

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).query('`Missing Count` > 0')

## Part 2: Selecting, Filtering, Sorting

In [ ]:
# Select a single column (returns Series)
ages = df['Age']
print(type(ages))
print(ages.describe())

In [ ]:
# Select multiple columns (returns DataFrame)
df[['Name', 'Age', 'Sex', 'Survived']].head()

In [ ]:
# Filter rows — female passengers who survived
survivors_female = df[(df['Sex'] == 'female') & (df['Survived'] == 1)]
print(f'Female survivors: {len(survivors_female)}')
survivors_female[['Name', 'Age', 'Pclass', 'Fare']].head()

In [ ]:
# .query() is often more readable
first_class_adults = df.query('Pclass == 1 and Age >= 18')
print(f'First class adults: {len(first_class_adults)}')

# Sort by Fare descending
top_payers = df.nlargest(5, 'Fare')[['Name', 'Pclass', 'Fare', 'Survived']]
top_payers

In [ ]:
# .loc vs .iloc
# .loc uses LABELS (index values, column names)
print('Row 0, Age and Sex:')
print(df.loc[0, ['Age', 'Sex']])

# .iloc uses INTEGER positions
print('\nFirst 3 rows, columns 2-4:')
df.iloc[:3, 2:5]

## Part 3: GroupBy Aggregations

GroupBy is one of the most powerful pandas operations. The pattern is always:
```
df.groupby('column')[other_columns].aggregate_function()
```

In [ ]:
# Survival rate by Pclass
survival_by_class = df.groupby('Pclass')['Survived'].agg(['sum', 'count', 'mean'])
survival_by_class.columns = ['Survivors', 'Total', 'Survival Rate']
survival_by_class['Survival Rate'] = (survival_by_class['Survival Rate'] * 100).round(1)
survival_by_class

In [ ]:
# Multiple aggregations with named outputs
df.groupby(['Pclass', 'Sex']).agg(
    n_passengers=('PassengerId', 'count'),
    survival_rate=('Survived', 'mean'),
    avg_age=('Age', 'mean'),
    avg_fare=('Fare', 'mean'),
).round(2)

In [ ]:
# Pivot table — same info, different shape
pivot = df.pivot_table(
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean',
)
print('Survival rate by class and sex:')
(pivot * 100).round(1).style.format('{:.1f}%').background_gradient(cmap='RdYlGn')

## Part 4: Handling Missing Values

In [ ]:
# Strategy depends on the column:
# Age: impute with median (skewed distribution)
# Embarked: impute with mode (most common)
# Cabin: too many missing — create binary feature instead

df_clean = df.copy()

# Fill Age with median
age_median = df_clean['Age'].median()
df_clean['Age'].fillna(age_median, inplace=True)
print(f'Age median: {age_median}')

# Fill Embarked with mode
embarked_mode = df_clean['Embarked'].mode()[0]
df_clean['Embarked'].fillna(embarked_mode, inplace=True)

# Create HasCabin binary feature
df_clean['HasCabin'] = df_clean['Cabin'].notna().astype(int)

# Verify
print(f'\nRemaining missing values:')
print(df_clean[['Age', 'Embarked', 'HasCabin']].isnull().sum())

## Part 5: Feature Engineering

In [ ]:
# Extract title from Name
df_clean['Title'] = df_clean['Name'].str.extract(r',\s*(\w+)\.', expand=False)

print('Title counts:')
print(df_clean['Title'].value_counts())

# Group rare titles
RARE_TITLES = {'Dr', 'Rev', 'Col', 'Major', 'Mlle', 'Countess', 'Ms',
               'Lady', 'Jonkheer', 'Don', 'Mme', 'Capt', 'Sir'}
df_clean['Title'] = df_clean['Title'].apply(
    lambda x: x if x not in RARE_TITLES else 'Rare'
)

print('\nCleaned title counts:')
print(df_clean['Title'].value_counts())

In [ ]:
# Family features
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)

# Survival rate by family size
family_survival = df_clean.groupby('FamilySize')['Survived'].agg(['mean', 'count'])
family_survival.columns = ['Survival Rate', 'Count']
family_survival['Survival Rate'] = (family_survival['Survival Rate'] * 100).round(1)

print('Survival by family size:')
print(family_survival)

In [ ]:
# Age groups
df_clean['AgeGroup'] = pd.cut(
    df_clean['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'YoungAdult', 'MiddleAge', 'Senior']
)

age_survival = df_clean.groupby('AgeGroup')['Survived'].mean().round(3)
print('Survival rate by age group:')
print((age_survival * 100).round(1))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
age_survival.plot(kind='bar', ax=axes[0], color='steelblue', title='Survival Rate by Age Group')
axes[0].set_ylabel('Survival Rate')
axes[0].tick_params(rotation=45)

df_clean.groupby('Title')['Survived'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral', title='Survival Rate by Title'
)
plt.tight_layout()
plt.show()

## Part 6: Merging DataFrames

In [ ]:
# Simulate a second DataFrame (class descriptions)
class_info = pd.DataFrame({
    'Pclass': [1, 2, 3],
    'ClassName': ['First Class', 'Second Class', 'Third Class'],
    'MedianFare2025': [350, 150, 45],  # Hypothetical modern equivalent
})

# Merge types:
# inner: only rows with matching keys in BOTH
# left:  all rows from LEFT, NaN for unmatched RIGHT
# right: all rows from RIGHT, NaN for unmatched LEFT
# outer: all rows from BOTH, NaN for unmatched

df_merged = df_clean.merge(class_info, on='Pclass', how='left')
print(f'Shape before: {df_clean.shape}')
print(f'Shape after merge: {df_merged.shape}')
df_merged[['Name', 'Pclass', 'ClassName', 'MedianFare2025']].head()

In [ ]:
# Concatenating DataFrames (stacking rows)
survivors = df_clean[df_clean['Survived'] == 1].copy()
non_survivors = df_clean[df_clean['Survived'] == 0].copy()
print(f'Survivors: {len(survivors)}, Non-survivors: {len(non_survivors)}')

# Recombine (this is trivial but shows concat)
recombined = pd.concat([survivors, non_survivors], ignore_index=True)
print(f'Recombined: {len(recombined)} rows')

## Summary — What You Learned

| Operation | Method |
|-----------|--------|
| Load CSV | `pd.read_csv()` |
| Inspect | `.head()`, `.info()`, `.describe()` |
| Select column | `df['col']` or `df.col` |
| Select multiple | `df[['col1', 'col2']]` |
| Filter rows | `df[condition]` or `df.query()` |
| Sort | `df.sort_values()`, `df.nlargest()` |
| Label indexing | `df.loc[row, col]` |
| Position indexing | `df.iloc[row_pos, col_pos]` |
| GroupBy | `df.groupby('col').agg()` |
| Fill missing | `df['col'].fillna(value)` |
| Create new col | `df['new'] = expression` |
| Merge | `df1.merge(df2, on='key')` |
| Stack rows | `pd.concat([df1, df2])` |

## Your Challenge

Extend this notebook:
1. Create a `FareBand` feature using `pd.qcut` (quartile-based bins)
2. Compute survival rate for each combination of `Sex` + `AgeGroup`
3. Find the passenger with the highest fare who did NOT survive
4. Create a summary table: mean, median, and std of Fare grouped by Title